### Multimodal fusion with leave-one-donor-out validation

Tests whether adding RNA to protein improves cell type prediction when the model is validated on donors it has never seen.

The `notebooks/logistic.ipynb` notebook used a random 80/20 split. Cells from the same donor look alike, so a random split can be too optimistic. Here every donor takes a turn as the held-out set.

The model code (logistic regression helper and the two-branch MLP) lives in `src/fusion.py`. Everything else is in here.

In [ ]:
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, f1_score

sys.path.append("../src")
from load_data import load_split, load_sample_submission
from fusion import CLASSES, fit_predict_lr, fit_predict_mlp

In [2]:
# load the train split and the two test splits
cells, rna, protein, labels = load_split("train")
test1_cells, test1_rna, test1_protein, _ = load_split("test1")
test2_cells, test2_rna, test2_protein, _ = load_split("test2")
sample_submission = load_sample_submission()

# plain numpy arrays: protein, RNA (sparse -> dense), labels as numbers 0-4, and the donor of each cell
P = protein.values.astype(np.float32)
R = rna.toarray().astype(np.float32)
y = labels["cell_type"].map({name: number for number, name in enumerate(CLASSES)}).values
donors = cells["donor"].values

print("protein:", P.shape, " RNA:", R.shape, " labels:", y.shape)
print("class order:", CLASSES)

protein: (7000, 217)  RNA: (7000, 3000)  labels: (7000,)
class order: ['B', 'CD4 T', 'CD8 T', 'Mono', 'NK']


### Cell types per donor

There are 8 donors. Each cell type shows up in every donor, but in different amounts.

In [3]:
# cells per cell type in each donor
cells.groupby("donor")["cell_type"].value_counts().unstack()

cell_type,B,CD4 T,CD8 T,Mono,NK
donor,,,,,
P1,34,335,158,261,70
P2,66,290,171,179,76
P3,45,214,90,190,116
P4,74,270,144,247,37
P5,76,134,154,379,120
P6,156,220,88,327,136
P7,164,248,141,452,90
P8,42,295,271,318,122


### Leave-one-donor-out cross-validation

For each donor: train on the other 7 donors, predict the held-out donor. After 8 rounds every cell has one prediction from a model that never saw its donor.

Models compared (scalers are fit on the training donors only, so nothing leaks):
- LR protein / RNA / combined: logistic regression (combined = RNA and protein columns side by side)
- MLP protein / RNA / fusion: small neural network. The fusion MLP has one branch for protein and one for RNA, joined before the final decision. Loss is class-weighted because the cell types are imbalanced.

In [4]:
# each model takes the training rows and the test rows and returns class probabilities for the test rows
def lr_protein(train, test):
    return fit_predict_lr(P[train], y[train], [P[test]])[0]

def lr_rna(train, test):
    return fit_predict_lr(R[train], y[train], [R[test]], C=0.01)[0]

def lr_combined(train, test):
    both_train = np.hstack([P[train], R[train]])
    both_test = np.hstack([P[test], R[test]])
    return fit_predict_lr(both_train, y[train], [both_test], C=0.01)[0]

def mlp_protein(train, test):
    return fit_predict_mlp(P[train], R[train], y[train], [P[test]], [R[test]], use_rna=False)[0]

def mlp_rna(train, test):
    return fit_predict_mlp(P[train], R[train], y[train], [P[test]], [R[test]], use_protein=False)[0]

def mlp_fusion(train, test):
    return fit_predict_mlp(P[train], R[train], y[train], [P[test]], [R[test]])[0]

models = {
    "LR protein": lr_protein,
    "LR RNA": lr_rna,
    "LR combined": lr_combined,
    "MLP protein": mlp_protein,
    "MLP RNA": mlp_rna,
    "MLP fusion": mlp_fusion,
}

In [5]:
# probs[model] holds one row of 5 class probabilities per cell, filled in one donor at a time
probs = {name: np.zeros((len(y), 5)) for name in models}
per_donor_rows = []

for donor in sorted(set(donors)):
    test_rows = np.where(donors == donor)[0]
    train_rows = np.where(donors != donor)[0]

    for name, model_function in models.items():
        probs[name][test_rows] = model_function(train_rows, test_rows)
        predictions = probs[name][test_rows].argmax(axis=1)
        per_donor_rows.append({
            "donor": donor,
            "model": name,
            "wrong cells": int((predictions != y[test_rows]).sum()),
        })
    print("finished held-out donor", donor)

finished held-out donor P1


finished held-out donor P2


finished held-out donor P3


finished held-out donor P4


finished held-out donor P5


finished held-out donor P6


finished held-out donor P7


finished held-out donor P8


### Late fusion and model comparison

Two more candidates that need no new training: late fusion averages the protein-model and RNA-model probabilities.

In [6]:
probs["LR late fusion"] = (probs["LR protein"] + probs["LR RNA"]) / 2
probs["MLP late fusion"] = (probs["MLP protein"] + probs["MLP RNA"]) / 2

# score every model over all 7,000 cells (each predicted while its donor was held out)
summary_rows = []
for name, model_probs in probs.items():
    predictions = model_probs.argmax(axis=1)
    donor_f1 = []
    for donor in sorted(set(donors)):
        mask = donors == donor
        donor_f1.append(f1_score(y[mask], predictions[mask], average="macro"))
    summary_rows.append({
        "model": name,
        "wrong cells": int((predictions != y).sum()),
        "accuracy": (predictions == y).mean(),
        "macro F1": f1_score(y, predictions, average="macro"),
        "worst-donor macro F1": min(donor_f1),
    })

summary = pd.DataFrame(summary_rows).set_index("model").sort_values("macro F1", ascending=False)
summary.round(4)

,wrong cells,accuracy,macro F1,worst-donor macro F1
model,,,,
LR protein,15,0.9979,0.9975,0.9952
LR late fusion,20,0.9971,0.9968,0.9905
MLP protein,21,0.9970,0.9964,0.9925
LR combined,23,0.9967,0.9961,0.9931
MLP late fusion,30,0.9957,0.9952,0.9882
MLP fusion,28,0.9960,0.9952,0.9817
LR RNA,118,0.9831,0.9825,0.9730
MLP RNA,183,0.9739,0.9725,0.9571


`wrong cells` is out of 7,000. Protein-only logistic regression is the best single model. The fusion MLP and the concatenated logistic regression are not better. RNA alone is clearly worse. The MLP got no tuning (30 epochs, default settings), so it could improve a little, but the error analysis below suggests there is not much room.

In [7]:
# wrong cells for each held-out donor
pd.DataFrame(per_donor_rows).pivot(index="donor", columns="model", values="wrong cells")

model,LR RNA,LR combined,LR protein,MLP RNA,MLP fusion,MLP protein
donor,,,,,,
P1,18,4,3,30,9,5
P2,16,2,3,24,4,3
P3,8,1,1,13,3,2
P4,23,4,2,37,3,4
P5,9,1,2,12,2,2
P6,9,1,1,8,1,1
P7,12,4,0,23,2,1
P8,23,6,3,36,4,3


### Error analysis of the protein model

In [8]:
protein_probs = probs["LR protein"]
rna_probs = probs["LR RNA"]
protein_wrong = protein_probs.argmax(axis=1) != y
print("protein errors:", protein_wrong.sum(), "of", len(y))

# rows = true type, columns = predicted type
pd.DataFrame(confusion_matrix(y, protein_probs.argmax(axis=1)), index=CLASSES, columns=CLASSES)

protein errors: 15 of 7000


,B,CD4 T,CD8 T,Mono,NK
B,656,1,0,0,0
CD4 T,0,2002,4,0,0
CD8 T,0,3,1211,0,3
Mono,0,0,0,2353,0
NK,0,2,2,0,763


In [9]:
confidence = protein_probs.max(axis=1)
print("protein confidence on the wrong cells:", np.round(np.sort(confidence[protein_wrong]), 2))
print("RNA-only model is right on", (rna_probs.argmax(axis=1)[protein_wrong] == y[protein_wrong]).sum(), "of these", protein_wrong.sum(), "cells")
print("cells with protein confidence < 0.9:", (confidence < 0.9).sum(), "- errors among them:", (protein_wrong & (confidence < 0.9)).sum())

protein confidence on the wrong cells: [0.41 0.5  0.57 0.63 0.68 0.71 0.83 0.85 0.9  0.94 0.96 0.99 1.   1.
 1.  ]
RNA-only model is right on 4 of these 15 cells
cells with protein confidence < 0.9: 71 - errors among them: 9


About 3 errors are confident (>= 0.99) and wrong, which is what a mislabeled cell would look like. RNA only rescues 4 of the 15. So the protein model is close to the ceiling on this data, and fusion has little left to fix.

### RNA weight in late fusion

Blend = (1 - w) x protein probabilities + w x RNA probabilities.

In [10]:
weight_rows = []
for rna_weight in [0, 0.1, 0.2, 0.3, 0.4, 0.5]:
    blend = (1 - rna_weight) * protein_probs + rna_weight * rna_probs
    weight_rows.append({
        "RNA weight": rna_weight,
        "wrong cells": int((blend.argmax(axis=1) != y).sum()),
        "macro F1": round(f1_score(y, blend.argmax(axis=1), average="macro"), 4),
    })
pd.DataFrame(weight_rows)

,RNA weight,wrong cells,macro F1
0,0.0,15,0.9975
1,0.1,14,0.9977
2,0.2,13,0.9978
3,0.3,15,0.9975
4,0.4,16,0.9974
5,0.5,20,0.9968


A small RNA weight (0.2) gives 13 wrong cells vs 15 for protein alone. That is a 2-cell difference picked on the same CV, so treat it as a tie, not a win. We use 0.2 because it is multimodal and at least as good as protein alone, not because 2 cells matter.

### Bias check by donor, time point, and cell quality

In [11]:
final_rna_weight = 0.2
blend = (1 - final_rna_weight) * protein_probs + final_rna_weight * rna_probs

meta = cells[["donor", "time", "pct_counts_mt", "n_genes_by_counts", "total_counts"]].copy()
meta["wrong"] = blend.argmax(axis=1) != y

print("wrong cells by donor:")
print(meta.groupby("donor")["wrong"].sum().to_frame().T)
print()
print("error rate by time point:")
print(meta.groupby("time")["wrong"].mean().round(4).to_frame().T)
print()
print("median quality metrics, correct vs wrong cells:")
print(meta.groupby("wrong")[["pct_counts_mt", "n_genes_by_counts", "total_counts"]].median().round(1))

wrong cells by donor:
donor  P1  P2  P3  P4  P5  P6  P7  P8
wrong   3   2   0   2   2   1   0   3

error rate by time point:
time        0       3       7
wrong  0.0013  0.0021  0.0022

median quality metrics, correct vs wrong cells:
       pct_counts_mt  n_genes_by_counts  total_counts
wrong                                                
False            5.6             2056.0        6966.0
True             4.9             2573.0        8073.0


Errors are spread across donors and time points, and no single donor drives them. Wrong cells do not have worse quality metrics. The test sets contain the same 8 donors as train, so this donor-held-out check is stricter than the test sets themselves.

### Final submission

Train protein logistic regression (C = 0.1) and RNA logistic regression (C = 0.01) on all 7,000 cells, blend their probabilities 0.8 / 0.2, and predict both test sets.

Public leaderboard score (Test Set 1, macro F1): 0.99792, compared with 0.99730 for the protein-only logistic regression submission.

In [12]:
# use the same protein columns, in the same order, as in training
test_P = [test1_protein[protein.columns].values.astype(np.float32), test2_protein[protein.columns].values.astype(np.float32)]
test_R = [test1_rna.toarray().astype(np.float32), test2_rna.toarray().astype(np.float32)]

# probabilities for [test1, test2] from each model (scaler is fit on the 7,000 training cells inside fit_predict_lr)
final_protein_probs = fit_predict_lr(P, y, test_P, C=0.1)
final_rna_probs = fit_predict_lr(R, y, test_R, C=0.01)

test1_blend = (1 - final_rna_weight) * final_protein_probs[0] + final_rna_weight * final_rna_probs[0]
test2_blend = (1 - final_rna_weight) * final_protein_probs[1] + final_rna_weight * final_rna_probs[1]

# one table per test set, test1 on top of test2
test1_submission = pd.DataFrame({"ID": test1_cells["cell_id"].values, "label": test1_blend.argmax(axis=1)})
test2_submission = pd.DataFrame({"ID": test2_cells["cell_id"].values, "label": test2_blend.argmax(axis=1)})
submission = pd.concat([test1_submission, test2_submission], ignore_index=True)
print(submission.shape)

(3000, 2)


In [13]:
# the label order must be B, CD4 T, CD8 T, Mono, NK = 0-4
print("label order is correct:", CLASSES == ["B", "CD4 T", "CD8 T", "Mono", "NK"])

# check the submission matches the sample format
print("same columns:", list(submission.columns) == list(sample_submission.columns))
print("same number of rows:", len(submission) == len(sample_submission))
print("same IDs in the same order:", (submission["ID"].values == sample_submission["ID"].values).all())
print("no duplicated IDs:", submission["ID"].duplicated().sum() == 0)
print("every label is 0-4:", submission["label"].isin([0, 1, 2, 3, 4]).all())

# share of each predicted label in each test set (a sanity check only, not for tuning)
print()
print("test1:")
print(test1_submission["label"].value_counts(normalize=True).sort_index().round(3))
print("test2:")
print(test2_submission["label"].value_counts(normalize=True).sort_index().round(3))

label order is correct: True
same columns: True
same number of rows: True
same IDs in the same order: True
no duplicated IDs: True
every label is 0-4: True

test1:
label
0    0.094
1    0.289
2    0.172
3    0.337
4    0.109
Name: proportion, dtype: float64
test2:
label
0    0.094
1    0.287
2    0.175
3    0.335
4    0.109
Name: proportion, dtype: float64


In [14]:
submission.to_csv("../results/submission_late_fusion.csv", index=False)

# how different is it from the protein logistic regression submission?
logreg_submission = pd.read_csv("../results/submission_protein_logreg.csv")
print("differs from the protein logreg submission on", (submission["label"].values != logreg_submission["label"].values).sum(), "of 3000 cells")

differs from the protein logreg submission on 3 of 3000 cells
